In [17]:
import cv2
import numpy as np
import pyrealsense2 as rs

# -----------------------------
# Camera and display parameters
# -----------------------------
COLOR_WIDTH = 640
COLOR_HEIGHT = 480
DEPTH_WIDTH = 640
DEPTH_HEIGHT = 480
FPS = 30

RGB_WINDOW = "RealSense RGB - click to query 3D point"
DEPTH_WINDOW = "RealSense Depth Colormap"

# Median depth window around the clicked pixel.
MEDIAN_WINDOW_SIZE = 7

# Depth visualization range. This only affects the displayed colormap,
# not the depth value used for 3D deprojection.
DEPTH_VIS_MIN_M = 0.15
DEPTH_VIS_MAX_M = 0.45

# The mouse callback reads these latest aligned frames. 
latest_aligned_depth_frame = None 
latest_color_intrinsics = None 
last_click_result = None

def get_median_depth_around_pixel(depth_frame, u, v, window_size=7):
    """Return median valid depth and valid count in a local pixel window."""
    half_size = window_size // 2
    width = depth_frame.get_width()
    height = depth_frame.get_height()

    x_min = max(0, u - half_size)
    x_max = min(width - 1, u + half_size)
    y_min = max(0, v - half_size)
    y_max = min(height - 1, v + half_size)

    valid_depths = []
    for y in range(y_min, y_max + 1):
        for x in range(x_min, x_max + 1):
            depth_m = depth_frame.get_distance(x, y)
            if depth_m > 0 and np.isfinite(depth_m):
                valid_depths.append(depth_m)

    valid_count = len(valid_depths)
    if valid_count == 0:
        return None, 0

    return float(np.median(valid_depths)), valid_count

def deproject_pixel_to_camera_point(intrinsics, u, v, depth_m):
    """Convert a color pixel and metric depth to camera-frame XYZ in meters."""
    x_m, y_m, z_m = rs.rs2_deproject_pixel_to_point(
        intrinsics,
        [float(u), float(v)],
        float(depth_m),
    )
    return float(x_m), float(y_m), float(z_m)

def on_mouse(event, u, v, flags, userdata):
    """Handle left-click on the RGB image and print the corresponding 3D point."""
    global latest_aligned_depth_frame, latest_color_intrinsics, last_click_result

    if event != cv2.EVENT_LBUTTONDOWN:
        return

    if latest_aligned_depth_frame is None or latest_color_intrinsics is None:
        print("No aligned frames available yet.")
        return

    # Because depth is aligned to color, this RGB pixel coordinate can be used
    # directly on the aligned depth frame.
    single_depth_m = latest_aligned_depth_frame.get_distance(u, v)
    single_depth_valid = single_depth_m > 0 and np.isfinite(single_depth_m)

    median_depth_m, valid_count = get_median_depth_around_pixel(
        latest_aligned_depth_frame,
        u,
        v,
        window_size=MEDIAN_WINDOW_SIZE,
    )
    median_depth_valid = median_depth_m is not None

    single_point_m = None
    median_point_m = None

    # RealSense returns camera coordinates in meters when depth is in meters.
    if single_depth_valid:
        single_point_m = deproject_pixel_to_camera_point(
            latest_color_intrinsics,
            u,
            v,
            single_depth_m,
        )
    if median_depth_valid:
        median_point_m = deproject_pixel_to_camera_point(
            latest_color_intrinsics,
            u,
            v,
            median_depth_m,
        )

    print(f"pixel coordinate: ({u}, {v})")
    if single_depth_valid:
        print(f"single pixel depth: {single_depth_m:.6f} meter")
    else:
        print("single pixel depth: invalid")

    if median_depth_valid:
        print(f"median depth in {MEDIAN_WINDOW_SIZE}x{MEDIAN_WINDOW_SIZE} window: {median_depth_m:.6f} meter")
    else:
        print("invalid median depth")
    print(f"valid depth pixel count: {valid_count}")

    if single_point_m is not None:
        x_m, y_m, z_m = single_point_m
        print(f"single-pixel 3D camera coordinate: X={x_m:.6f}, Y={y_m:.6f}, Z={z_m:.6f} meter")
    else:
        print("single-pixel 3D camera coordinate: invalid")

    if median_point_m is not None:
        x_m, y_m, z_m = median_point_m
        print(f"median-depth 3D camera coordinate: X={x_m:.6f}, Y={y_m:.6f}, Z={z_m:.6f} meter")
    else:
        print("median-depth 3D camera coordinate: invalid")
    print("-" * 50)

    last_click_result = {
        "pixel": (u, v),
        "single_depth_m": float(single_depth_m) if single_depth_valid else None,
        "single_point_m": single_point_m,
        "median_depth_m": median_depth_m,
        "median_point_m": median_point_m,
        "valid_count": valid_count,
        "valid": median_depth_valid,
    }

def draw_last_click(color_image):
    """Draw the last clicked pixel and its 3D coordinate on the RGB image."""
    if last_click_result is None:
        return color_image

    display = color_image.copy()
    u, v = last_click_result["pixel"]
    cv2.circle(display, (u, v), 5, (0, 0, 255), -1) 
    cv2.circle(display, (u, v), 12, (0, 0, 255), 2)

    if last_click_result["median_point_m"] is not None:
        x_m, y_m, z_m = last_click_result["median_point_m"]
        text_lines = [
            f"pixel: ({u}, {v})",
            f"median depth: {last_click_result['median_depth_m']:.4f} m",
            f"median XYZ: ({x_m:.4f}, {y_m:.4f}, {z_m:.4f}) m",
            f"valid pixels: {last_click_result['valid_count']}",
        ]
    else:
        text_lines = [f"pixel: ({u}, {v})", "invalid median depth"]

    x0, y0 = 10, 25
    for i, text in enumerate(text_lines):
        y = y0 + i * 24
        cv2.putText(display, text, (x0, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 3, cv2.LINE_AA)
        cv2.putText(display, text, (x0, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 1, cv2.LINE_AA)

    return display

def make_depth_colormap(depth_frame):
    """Create a near-range depth colormap while keeping invalid depth black."""
    depth_image = np.asanyarray(depth_frame.get_data())
    depth_m = depth_image.astype(np.float32) * depth_frame.get_units()
    valid_mask = depth_image > 0

    depth_normalized = (depth_m - DEPTH_VIS_MIN_M) / (DEPTH_VIS_MAX_M - DEPTH_VIS_MIN_M)
    depth_8bit = np.clip(depth_normalized * 255.0, 0, 255).astype(np.uint8)
    depth_colormap = cv2.applyColorMap(depth_8bit, cv2.COLORMAP_JET)
    depth_colormap[~valid_mask] = (0, 0, 0)

    return depth_colormap

def main():
    global latest_aligned_depth_frame, latest_color_intrinsics

    pipeline = rs.pipeline() 
    config = rs.config()

    config.enable_stream(rs.stream.color, COLOR_WIDTH, COLOR_HEIGHT, rs.format.bgr8, FPS)
    config.enable_stream(rs.stream.depth, DEPTH_WIDTH, DEPTH_HEIGHT, rs.format.z16, FPS)

    # This align object is the key step: it aligns every depth frame to color.
    align_to_color = rs.align(rs.stream.color)

    cv2.namedWindow(RGB_WINDOW, cv2.WINDOW_AUTOSIZE) 
    cv2.namedWindow(DEPTH_WINDOW, cv2.WINDOW_AUTOSIZE) 
    cv2.setMouseCallback(RGB_WINDOW, on_mouse)

    print("Starting RealSense pipeline...") 
    print("Left-click the RGB window to query 3D point. Press q to quit.")

    try:
        profile = pipeline.start(config)

        # Warm up auto-exposure for a few frames.
        for _ in range(15):
            pipeline.wait_for_frames()

        while True: 
            frames = pipeline.wait_for_frames()

            # Always use aligned_frames after this point.
            aligned_frames = align_to_color.process(frames)
            aligned_depth_frame = aligned_frames.get_depth_frame()
            color_frame = aligned_frames.get_color_frame()

            if not aligned_depth_frame or not color_frame:
                continue

            latest_aligned_depth_frame = aligned_depth_frame
            latest_color_intrinsics = color_frame.profile.as_video_stream_profile().intrinsics

            color_image = np.asanyarray(color_frame.get_data())
            depth_colormap = make_depth_colormap(aligned_depth_frame)

            color_display = draw_last_click(color_image)

            cv2.imshow(RGB_WINDOW, color_display) 
            cv2.imshow(DEPTH_WINDOW, depth_colormap)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    finally:
        print("Stopping RealSense pipeline...") 
        pipeline.stop() 
        cv2.destroyAllWindows()

main()




Starting RealSense pipeline...
Left-click the RGB window to query 3D point. Press q to quit.
Stopping RealSense pipeline...
